# 02 Scrub – Studentenwohnheime (Bereinigung + NRW-Filter)

Ziel dieses Notebooks:
- Load: `data/processed/student_housing_de_obtain.csv`
- Scrub: Werte bereinigen (z. B. "-", Fußnoten, Strings → Zahlen)
- NRW-Filter über eine definierte Städtemenge (Hochschulorte in NRW)
- Save: `data/processed/student_housing_nrw_scrub.csv`



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

path_in = Path("../../data/processed/student_housing_de_obtain.csv")
path_in.exists(), path_in.resolve()


C:\Users\User\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/student_housing_de_obtain.csv'))

In [2]:
df = pd.read_csv(path_in)
df.shape, df.head(5)


((239, 5),
          hochschulort wohnheime_2025_anzahl wohnheimplaetze_2025  \
 0               Aalen                     7                  531   
 1            Albstadt                     2                   82   
 2  Bad Mergentheim 4)                     2                   35   
 3            Biberach                     1                   64   
 4        Esslingen 5)                     5                  857   
 
   studierende_ws_2024_2025 studierende_je_wohnheimplatz_2025  
 0                     4117                                 8  
 1                     1392                                17  
 2                      550                                16  
 3                     2122                                33  
 4                     4807                                 6  )

In [3]:
def to_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()

    if s in {"-", "–", "—", ""}:
        return np.nan

    s = re.sub(r"[^\d,\.]", "", s)

    s = s.replace(".", "").replace(",", ".")

    if s == "":
        return np.nan

    try:
        return float(s)
    except ValueError:
        return np.nan

num_cols = [
    "wohnheime_2025_anzahl",
    "wohnheimplaetze_2025",
    "studierende_ws_2024_2025",
    "studierende_je_wohnheimplatz_2025",
]

df_clean = df.copy()

for c in num_cols:
    df_clean[c] = df_clean[c].apply(to_number)

df_clean[num_cols].dtypes, df_clean[num_cols].isna().sum()


(wohnheime_2025_anzahl                float64
 wohnheimplaetze_2025                 float64
 studierende_ws_2024_2025             float64
 studierende_je_wohnheimplatz_2025    float64
 dtype: object,
 wohnheime_2025_anzahl                39
 wohnheimplaetze_2025                 37
 studierende_ws_2024_2025             35
 studierende_je_wohnheimplatz_2025    37
 dtype: int64)

In [4]:
def clean_city_name(x):
    if pd.isna(x):
        return x
    s = str(x).strip()
    s = re.sub(r"\s*\d+\)\s*$", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

df_clean["hochschulort"] = df_clean["hochschulort"].apply(clean_city_name)

df_clean["hochschulort"].head(10)


0              Aalen
1           Albstadt
2    Bad Mergentheim
3           Biberach
4          Esslingen
5           Freiburg
6    Friedrichshafen
7         Furtwangen
8         Geislingen
9          Göppingen
Name: hochschulort, dtype: object

In [5]:
df_clean["wohnheimplatzrelation"] = 1 / df_clean["studierende_je_wohnheimplatz_2025"]

df_clean[["hochschulort"] + num_cols + ["wohnheimplatzrelation"]].head(5)


,hochschulort,wohnheime_2025_anzahl,wohnheimplaetze_2025,studierende_ws_2024_2025,studierende_je_wohnheimplatz_2025,wohnheimplatzrelation
0,Aalen,7.0,531.0,4117.0,8.0,0.125000
1,Albstadt,2.0,82.0,1392.0,17.0,0.058824
2,Bad Mergentheim,2.0,35.0,550.0,16.0,0.062500
3,Biberach,1.0,64.0,2122.0,33.0,0.030303
4,Esslingen,5.0,857.0,4807.0,6.0,0.166667


In [6]:
nrw_cities = sorted([
    "Aachen","Bielefeld","Bochum","Bonn","Dortmund","Duisburg","Düsseldorf",
    "Essen","Hagen","Köln","Münster","Paderborn","Siegen","Wuppertal",
    "Gelsenkirchen","Krefeld","Mönchengladbach"
])

df_nrw = df_clean[df_clean["hochschulort"].isin(nrw_cities)].copy()

df_nrw.shape, df_nrw["hochschulort"].sort_values().unique()


((17, 6),
 array(['Aachen', 'Bielefeld', 'Bochum', 'Bonn', 'Dortmund', 'Duisburg',
        'Düsseldorf', 'Essen', 'Gelsenkirchen', 'Hagen', 'Krefeld', 'Köln',
        'Mönchengladbach', 'Münster', 'Paderborn', 'Siegen', 'Wuppertal'],
       dtype=object))

In [7]:
out_path = Path("../../data/processed/student_housing_nrw_scrub.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_nrw.to_csv(out_path, index=False)

out_path.exists(), out_path.resolve()


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/student_housing_nrw_scrub.csv'))

## Scrub-Ergebnis

- `student_housing_de_obtain.csv` wurde bereinigt (Zahlen konvertiert, Fußnoten entfernt).
- NRW-Hochschulorte wurden gefiltert.
- Ergebnis wurde gespeichert als `data/processed/student_housing_nrw_scrub.csv`.

